# 📝 Thử nghiệm Query Rewriting (Viết lại câu truy vấn)
**Mục tiêu:** Kiểm nghiệm module `query_generator` từ `traffic_prompts.yaml`.

Module này nhận câu hỏi tự nhiên của người dùng (ngôn ngữ thông thường) và sinh ra 3 câu truy vấn tìm kiếm chuyên biệt hơn, giàu từ khóa pháp lý hơn.

**Điều cần kiểm tra:**
1. Chất lượng các câu rewrite (có đúng ngữ nghĩa không?)
2. Mức độ phong phú từ khóa pháp lý
3. So sánh điểm retrieval trước vs sau rewriting

In [ ]:
import os, sys, yaml, json
from dotenv import load_dotenv

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

import google.generativeai as genai
from rank_bm25 import BM25Okapi
from source.core.config import Settings
from source.retrieval.hybrid_retriever import HybridRetriever

settings = Settings()
api_key = settings.api_key or os.getenv('API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.0-flash')

PROMPT_PATH = os.path.join(PROJECT_ROOT, 'source', 'core', 'traffic_prompts.yaml')
with open(PROMPT_PATH, 'r', encoding='utf-8') as f:
    prompts = yaml.safe_load(f)['prompts']

CHUNKS_PATH = os.path.join(PROJECT_ROOT, 'Data', 'chunks', 'traffic_chunks.json')
retriever = HybridRetriever(settings=settings, collection_name="Traffic_Law_Hybrid")
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    retriever.corpus_chunks = json.load(f)
retriever.bm25 = BM25Okapi([retriever._tokenize(c['content']) for c in retriever.corpus_chunks])

print(f"✅ Sẵn sàng! {len(retriever.corpus_chunks):,} chunks trong BM25")

## 1. Hàm Query Rewriting

In [ ]:
def rewrite_query(query: str, history: str = "Trống") -> list:
    """Gọi query_generator prompt, trả về list 3 câu rewritten."""
    full_prompt = ""
    for msg in prompts['query_generator']['messages']:
        content = msg['content'].format(query=query, history=history)
        full_prompt += f"{'SYSTEM' if msg['role'] == 'system' else 'USER'}:\n{content}\n\n"
    response = model.generate_content(full_prompt)
    lines = [q.strip() for q in response.text.strip().split('\n') if q.strip()]
    return lines[:3]

# Test nhanh
sample = "xe máy uống bia bị phạt bao nhiêu?"
rewrites = rewrite_query(sample)
print(f"🔤 Câu gốc: {sample}")
print(f"\n✨ 3 câu rewritten:")
for i, r in enumerate(rewrites, 1):
    print(f"  [{i}] {r}")

## 2. So sánh chất lượng retrieval: Câu gốc vs Rewritten

In [ ]:
import numpy as np

def bm25_search(query: str, top_k: int = 5) -> list:
    tokenized_q = retriever._tokenize(query)
    scores = retriever.bm25.get_scores(tokenized_q)
    if np.max(scores) > 0:
        scores = scores / np.max(scores)
    top_indices = scores.argsort()[-top_k:][::-1]
    return [{"chunk": retriever.corpus_chunks[i], "score": float(scores[i])} for i in top_indices]

def search_with_rewriting(original_query: str, top_k: int = 5) -> list:
    """Tìm kiếm với Multi-Query (dùng rewritten queries), trả về tập hợp unique."""
    rewrites = rewrite_query(original_query)
    all_queries = [original_query] + rewrites
    
    seen = set()
    aggregate_results = []
    for q in all_queries:
        for r in bm25_search(q, top_k=top_k):
            key = r['chunk']['content'][:60]
            if key not in seen:
                seen.add(key)
                aggregate_results.append(r)
    
    return sorted(aggregate_results, key=lambda x: x['score'], reverse=True)[:top_k]

TEST_QUERIES_REWRITE = [
    {"q": "say rượu lái xe bị làm sao?", "expected": "nồng độ cồn"},
    {"q": "chạy nhanh quá bị dừng xe phạt gì?", "expected": "tốc độ"},
    {"q": "bằng lái không có thì làm sao?", "expected": "giấy phép lái xe"},
]

print("📊 SO SÁNH RETRIEVAL: Câu gốc vs Sau Rewriting\n")
for tc in TEST_QUERIES_REWRITE:
    print(f"{'='*65}")
    print(f"🔤 Câu gốc: '{tc['q']}'")
    
    # Retrieval câu gốc
    orig_results = bm25_search(tc['q'], top_k=3)
    print(f"\n  📋 Top-3 câu gốc:")
    for r in orig_results:
        print(f"     [{r['score']:.3f}] {r['chunk']['metadata']['dieu']} | {r['chunk']['content'][:70]}...")
    
    # Rewriting
    rewrites = rewrite_query(tc['q'])
    print(f"\n  ✨ Câu đã rewrite:")
    for r in rewrites:
        print(f"     → {r}")
    
    # Retrieval sau rewriting
    rewrite_results = search_with_rewriting(tc['q'], top_k=3)
    print(f"\n  📋 Top-3 sau rewriting:")
    for r in rewrite_results:
        hit = "✅" if tc['expected'] in r['chunk']['content'].lower() else "  "
        print(f"     {hit} [{r['score']:.3f}] {r['chunk']['metadata']['dieu']} | {r['chunk']['content'][:70]}...")
    print()

## 3. Đo lường Recall cải thiện sau rewriting

In [ ]:
def recall_at_k(results, expected_keyword, k=5):
    for r in results[:k]:
        content_lower = r['chunk']['content'].lower()
        if expected_keyword.lower() in content_lower:
            return 1.0
    return 0.0

recall_orig_list = []
recall_rw_list = []

print("📊 Recall@5: Câu gốc vs Sau Rewriting")
for tc in TEST_QUERIES_REWRITE:
    orig_results = bm25_search(tc['q'], top_k=5)
    rw_results = search_with_rewriting(tc['q'], top_k=5)
    
    r_orig = recall_at_k(orig_results, tc['expected'])
    r_rw = recall_at_k(rw_results, tc['expected'])
    recall_orig_list.append(r_orig)
    recall_rw_list.append(r_rw)
    
    delta = r_rw - r_orig
    change = "↑ Cải thiện" if delta > 0 else ("↓ Giảm" if delta < 0 else "= Không thay đổi")
    print(f"  '{tc['q'][:45]}'")
    print(f"     Gốc: {r_orig:.1f} → Rewrite: {r_rw:.1f}  {change}\n")

avg_orig = sum(recall_orig_list) / len(recall_orig_list)
avg_rw = sum(recall_rw_list) / len(recall_rw_list)
print(f"\n🏆 Recall@5 Trung bình:")
print(f"   Câu gốc    : {avg_orig:.3f}")
print(f"   Sau Rewrite: {avg_rw:.3f}  {'👍 Multi-query rewriting HIỆU QUẢ!' if avg_rw >= avg_orig else '⚠️  Cần cải thiện prompt rewriting'}")